# Step 2 — Embed: Dense + Sparse Vectors
- **Dense:** `all-mpnet-base-v2` (768-dim) via sentence-transformers  
- **Sparse:** BM25 term weights → Milvus sparse float format  
- **Output:** `embeddings.npz` (dense) + `sparse_rows.pkl` (sparse)

> ⚡ Enable GPU runtime: Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install -q sentence-transformers rank-bm25 scipy

from google.colab import drive
drive.mount('/content/drive')

import os, json
BASE_DIR   = '/content/drive/MyDrive/mdna_rag'
CHUNKS_IN  = f'{BASE_DIR}/chunks.jsonl'
DENSE_OUT  = f'{BASE_DIR}/embeddings_dense.npz'
SPARSE_OUT = f'{BASE_DIR}/embeddings_sparse.pkl'
print('✅ Ready')

In [ ]:
# ── Load all chunks ───────────────────────────────────────────────────────────
import json

chunks = []
with open(CHUNKS_IN) as f:
    for line in f:
        chunks.append(json.loads(line))

texts = [c['text'] for c in chunks]
print(f'Loaded {len(chunks):,} chunks')

In [ ]:
# ── Dense embeddings ─────────────────────────────────────────────────────────
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device='cuda')

BATCH = 256
dense_vecs = []

for i in tqdm(range(0, len(texts), BATCH), desc='Dense embedding'):
    batch = texts[i:i+BATCH]
    vecs  = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    dense_vecs.append(vecs)

dense_matrix = np.vstack(dense_vecs).astype('float32')  # shape (N, 768)
np.savez_compressed(DENSE_OUT, embeddings=dense_matrix)
print(f'✅ Dense matrix: {dense_matrix.shape}  →  {DENSE_OUT}')

In [ ]:
# ── Sparse vectors via BM25 ──────────────────────────────────────────────────
# Milvus expects sparse vectors as list of dicts: {token_id: weight}
import pickle, re
from rank_bm25 import BM25Okapi
from collections import defaultdict

# ── Finance-aware tokenizer ──────────────────────────────────────────────────
STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for',
    'of','with','as','by','from','is','was','are','were','be',
    'this','that','these','those','it','its','we','our','us',
    'have','had','has','will','would','could','should','may',
    'also','such','than','more','other','any','all','been',
}

def tokenize(text: str) -> list[str]:
    tokens = re.findall(r'[A-Za-z0-9][A-Za-z0-9\-\.]*', text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

print('Building corpus vocabulary...')
tokenized_corpus = [tokenize(t) for t in tqdm(texts, desc='Tokenizing')]

# Build vocabulary mapping token → int id
vocab: dict[str, int] = {}
for doc_tokens in tokenized_corpus:
    for tok in doc_tokens:
        if tok not in vocab:
            vocab[tok] = len(vocab)

print(f'Vocabulary size: {len(vocab):,}')

# Fit BM25
bm25 = BM25Okapi(tokenized_corpus)

# Convert to sparse dicts for each doc
sparse_vecs = []
for doc_tokens in tqdm(tokenized_corpus, desc='Building sparse vecs'):
    freq: dict[str, int] = defaultdict(int)
    for tok in doc_tokens:
        freq[tok] += 1

    sparse = {}
    for tok, tf in freq.items():
        idf = bm25.idf.get(tok, 0.0)
        if idf > 0:
            sparse[vocab[tok]] = float(tf * idf)   # BM25-weighted
    sparse_vecs.append(sparse)

with open(SPARSE_OUT, 'wb') as f:
    pickle.dump({'sparse_vecs': sparse_vecs, 'vocab': vocab}, f)

print(f'✅ Sparse vectors saved → {SPARSE_OUT}')
print(f'   Example non-zero dims: {len(sparse_vecs[0])}')

In [ ]:
# ── Checkpoint: verify alignment ─────────────────────────────────────────────
assert len(dense_vecs_flat := np.load(DENSE_OUT)['embeddings']) == len(chunks), \
    'Dense count mismatch'

with open(SPARSE_OUT,'rb') as f:
    sp = pickle.load(f)
assert len(sp['sparse_vecs']) == len(chunks), 'Sparse count mismatch'

print(f'✅ {len(chunks):,} chunks | dense {dense_vecs_flat.shape[1]}-dim | '
      f'vocab {len(sp["vocab"]):,} tokens')